In [14]:
"""
Run in Jupyter: this uses matplotlib + ipywidgets.
"""

import numpy as np
import datetime
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def simulate(
    c=5,
    battery_capacity_Wh=30,
    num_batteries=1,
    sleep_power_W=0.1,
    measuring_power_W=1,
    measuring_time_min=1,
    sending_power_W=10,
    sending_time_min=1,
    num_solar_cells=1,
    solar_spring_Wh=1,
    solar_summer_Wh=1.5,
    solar_autumn_Wh=1,
    solar_winter_Wh=0.8,
    simulation_days=30
):
    # --- Time arrays ---
    start_date = datetime.datetime(2025,1,1)
    minutes_per_day = 24*60
    total_minutes = simulation_days*minutes_per_day
    dates = [start_date + datetime.timedelta(minutes=i) for i in range(total_minutes)]
    hours = np.array([(d.hour + d.minute/60) for d in dates])

    # ----- Season index per day  (split into 4, leftover to last) -----
    days = np.arange(simulation_days)
    q1 = simulation_days // 4
    q2 = q1 * 2
    q3 = q1 * 3
    season_idx = np.zeros(simulation_days, dtype=int)
    season_idx[q1:q2] = 1
    season_idx[q2:q3] = 2
    season_idx[q3:]   = 3
    seasons = np.repeat(season_idx, minutes_per_day)
    season_solar = [solar_spring_Wh, solar_summer_Wh, solar_autumn_Wh, solar_winter_Wh]

    # --- Solar (fixed per season, but bell-shaped per day) ---
    mu, sigma = 14, 2
    solar_W = np.zeros(total_minutes)
    for s in (0,1,2,3):
        mask = (seasons==s)
        energy_per_day = season_solar[s]*num_solar_cells
        curve = np.exp(-0.5*((hours[mask]-mu)/sigma)**2)
        curve[(hours[mask]<7)|(hours[mask]>21)] = 0
        area = curve.sum()
        if area>0:
            curve = (curve/area)*(energy_per_day*60)
        solar_W[mask] = curve

    # --- Consumption schedule ---
    meas_W = np.zeros(total_minutes)
    send_W = np.zeros(total_minutes)
    sleep_W = np.zeros(total_minutes)

    cycle_len = measuring_time_min+sending_time_min
    for day in range(simulation_days):
        for cyc in range(c):
            start = day*minutes_per_day + cyc*cycle_len
            meas_W[start:start+measuring_time_min]=measuring_power_W
            send_W[start+measuring_time_min:start+cycle_len]=sending_power_W

    sleep_W[(meas_W==0)&(send_W==0)] = sleep_power_W
    total_cons = meas_W+send_W+sleep_W

    # --- battery integration ---
    batt = np.zeros(total_minutes)
    energy=0
    cap = battery_capacity_Wh*num_batteries
    for i in range(total_minutes):
        net = (solar_W[i]-total_cons[i])/60
        energy += net
        if energy>cap: energy=cap
        if energy<0: energy=0
        batt[i]=energy

    # ----- PLOTS -----
    plt.figure(figsize=(16,8))

    plt.title("Energiesimulation Datenboje")

    plt.subplot(2,1,1)
    plt.plot(dates,batt,label="Batterie (Wh)")
    plt.ylabel("Energie in [Wh]")
    plt.legend()

    plt.subplot(2,1,2)
    plt.plot(dates,solar_W,label="Solar (W)")
    plt.plot(dates,meas_W,label="Messung(W)")
    plt.plot(dates,send_W,label="Senden(W)")
    plt.plot(dates,sleep_W,label="Schlaf(W)")
    plt.ylabel("Energie in [W]")
    plt.legend()

    plt.xlabel("Datum")
    plt.tight_layout()
    plt.show()

# --- Widgets ---
interact(
    simulate,
    c=IntSlider(min=1,max=50,value=10,description='Cycles/day'),
    battery_capacity_Wh=FloatSlider(min=1,max=100,value=30,description='BattCap'),
    num_batteries=IntSlider(min=1,max=5,value=1,description='Batt'),
    sleep_power_W=FloatSlider(min=0.01,max=1,step=0.01,value=0.01,description='SleepW'),
    measuring_power_W=FloatSlider(min=0.1,max=10,value=1,description='MeasW'),
    measuring_time_min=IntSlider(min=1,max=30,value=1,description='MeasMin'),
    sending_power_W=FloatSlider(min=1,max=20,value=10,description='SendW'),
    sending_time_min=IntSlider(min=1,max=30,value=1,description='SendMin'),
    num_solar_cells=IntSlider(min=1,max=10,value=8,description='Cells'),
    solar_spring_Wh=FloatSlider(min=0.1,max=10,value=4,description='Spring'),
    solar_summer_Wh=FloatSlider(min=0.1,max=10,value=0.5,description='Summer'),
    solar_autumn_Wh=FloatSlider(min=0.1,max=10,value=1.5,description='Autumn'),
    solar_winter_Wh=FloatSlider(min=0.1,max=10,value=0.5,description='Winter'),
    simulation_days=IntSlider(min=1,max=365,value=32,description='Days')
)


interactive(children=(IntSlider(value=10, description='Cycles/day', max=50, min=1), FloatSlider(value=30.0, de…

<function __main__.simulate(c=5, battery_capacity_Wh=30, num_batteries=1, sleep_power_W=0.1, measuring_power_W=1, measuring_time_min=1, sending_power_W=10, sending_time_min=1, num_solar_cells=1, solar_spring_Wh=1, solar_summer_Wh=1.5, solar_autumn_Wh=1, solar_winter_Wh=0.8, simulation_days=30)>